# CICIDS2017 — Reliability Evaluation Framework (Final Robust Version)

This notebook is designed for a **zero-manual-setup Google Colab run**:
CICIDS2017 is downloaded automatically from Hugging Face, validated, processed in a
memory-conscious float32 pipeline, and evaluated under reference, temporal,
feature-missingness, combined-stress, degradation, interaction, attack-level,
and repeated-seed conditions.

**Temporal integrity is explicit:** if the selected CICIDS2017 mirror contains a
validated real row-level `Timestamp`, it is used. If the mirror is the standard
CICIDS2017 CSV variant without row-level timestamps, the notebook uses the
documented capture-day/source-file order (Monday→Friday). It never converts
`Flow Duration`, `Total TCP Flow Time`, or another feature into a fake timestamp.

The notebook fails early on incompatible schemas, unusable labels, invalid temporal
provenance, non-finite estimator inputs, or other missing experimental prerequisites.


### Finalized fixes in this version
- validated real `Timestamp` or explicit capture-day provenance; no flow-feature timestamp substitution;
- complete training partitions in `PAPER_MODE`;
- stratified bounded sampling only for fast validation;
- binary + multiclass evaluation;
- six-condition reliability matrix (Reference/Temporal × 0/10/20% missingness);
- split-level train-only imputer reuse for efficiency;
- model reuse across all stress conditions;
- degradation and temporal×missingness interaction analysis;
- attack-level recall across every reliability condition;
- five-seed aggregation with 95% t-based confidence intervals;
- automated completeness and scientific-validity checks before export.


In [ ]:
# 1. Configuration — final reliability experiment (optimized + validated)
from pathlib import Path
import os, gc, json, warnings, random, sys, subprocess
import numpy as np
import pandas as pd


# Hugging Face client — imported here before any downloader functions are defined.
try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub>=0.24.0"])
    from huggingface_hub import snapshot_download

SEED = 42
SEEDS = [42, 52, 62, 72, 82]

# ---------------- RUN MODES ----------------
# QUICK_VALIDATION: finishes much faster and verifies the entire pipeline.
# PAPER_MODE: uses the full verified training partitions and all five seeds.
QUICK_VALIDATION = True
PAPER_MODE = False
RUN_EXPERIMENTS = True

# Bounded validation size. This is NOT used in PAPER_MODE.
FAST_TRAIN_SAMPLES = 100_000
# In quick validation, preserve approximate class proportions rather than
# using one-example-per-class plus random fill. This setting is NEVER used
# in PAPER_MODE.
FAST_SAMPLING_STRATIFIED = True

# Exact test-time stress rates used by the reliability matrix.
MISSINGNESS_LEVELS = (0.0, 0.10, 0.20)

# Statistical reporting for the five-seed final experiment.
CI_LEVEL = 0.95

# Full-data paper protocol.
# Repeated seeds are expensive, so they run only in PAPER_MODE.
REPEATED_SEEDS = SEEDS

# ---------------- DATA / MEMORY ----------------
HF_SOURCE_PREFERENCE = ["kumaranimesh1901/CICIDS-2017", "bvk/CICIDS-2017", "San0160/CICIDS-2017"]

# Robust Hugging Face networking for Colab. snapshot_download resumes interrupted transfers.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")

# Never use these as temporal substitutes.
FORBIDDEN_TIMESTAMP_FEATURES = {
    "flow duration", "total tcp flow time", "total fwd packets",
    "total bwd packets", "flow iat mean", "flow iat std",
    "flow iat max", "flow iat min"
}

# Threading is intentionally conservative for shared Colab runtimes.
N_JOBS = max(1, min(2, (os.cpu_count() or 2)))
OUTPUT_DIR = Path("/content/reliability_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
random.seed(SEED)
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

FINAL_MODE = PAPER_MODE
if QUICK_VALIDATION and PAPER_MODE:
    raise ValueError("Choose either QUICK_VALIDATION=True or PAPER_MODE=True, not both.")

print("Configuration ready.")
print("HF source preference:", HF_SOURCE_PREFERENCE)
print("QUICK_VALIDATION:", QUICK_VALIDATION)
print("PAPER_MODE:", PAPER_MODE)
print("RUN_EXPERIMENTS:", RUN_EXPERIMENTS)
print("N_JOBS:", N_JOBS)


In [ ]:
# 2. Imports with a safe XGBoost fallback
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, classification_report
)

try:
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True
except Exception:
    XGB_AVAILABLE = False
    print("XGBoost is not currently available. Installing it...")
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "xgboost"])
    from xgboost import XGBClassifier
    XGB_AVAILABLE = True

print("Core imports loaded. XGBoost available:", XGB_AVAILABLE)


## 3. Dataset source — automatic Hugging Face import

The notebook follows the same automatic import pattern as the proven FAST v4 notebook:
`huggingface_hub.snapshot_download()` downloads CICIDS2017 directly into the Colab runtime.

No Google Drive, manual upload, or manually entered dataset path is required.

The loader accepts CICIDS2017 CSV variants with a valid `Label` column. If a genuine row-level
`Timestamp` is present and valid, it is used. If the selected source does not preserve a usable
row-level timestamp, temporal evaluation uses the documented CICIDS2017 capture-day/file order.

**Important:** `Flow Duration`, `Total TCP Flow Time`, packet counts, and other flow features are
never substituted for a timestamp.


In [ ]:
# 4. Automatic CICIDS2017 download — same proven pattern as the FAST v4 notebook
# No Google Drive, no manual upload, no manual dataset path.
#
# We intentionally use snapshot_download directly instead of HfApi/list_repo_files.
# This is more robust in Colab and avoids the HfApi import-order failure seen in v2/v3.
#
# Source preference:
#   1) local /content/CICIDS2017_timestamped if already present
#   2) kumaranimesh1901/CICIDS-2017
#   3) bvk/CICIDS-2017
#   4) San0160/CICIDS-2017
#
# A repository is accepted when it contains at least 5 CICIDS2017-style CSV files
# with a usable Label column. Timestamp is checked separately later.
# If no real row-level Timestamp exists, the notebook uses capture-day/file order.
# It NEVER substitutes Flow Duration / Total TCP Flow Time as a timestamp.

import time

try:
    from huggingface_hub import snapshot_download
except ImportError:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "huggingface_hub>=0.24.0"
    ])
    from huggingface_hub import snapshot_download

# Robust Hugging Face networking for Colab.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "60")

def normalize_header_name(value):
    return (
        str(value)
        .replace("\ufeff", "")
        .replace("\u00a0", " ")
        .strip()
        .casefold()
    )

def find_exact_column(columns, target):
    target_norm = normalize_header_name(target)
    return next(
        (c for c in columns if normalize_header_name(c) == target_norm),
        None
    )

def discover_csv_files(data_dir):
    data_dir = Path(data_dir)
    paths = sorted(
        list(data_dir.rglob("*.csv")) +
        list(data_dir.rglob("*.CSV"))
    )
    # Remove duplicates caused by case-insensitive filesystems.
    return list(dict.fromkeys(paths))

def inspect_cicids_files(csv_paths):
    usable = []
    diagnostics = []

    for path in csv_paths:
        try:
            head = pd.read_csv(
                path,
                nrows=0,
                encoding="cp1252",
                engine="c",
                low_memory=False,
            )
            label = find_exact_column(head.columns, "Label")
            timestamp = find_exact_column(head.columns, "Timestamp")

            if label is None:
                diagnostics.append({
                    "file": str(path),
                    "status": "Missing Label",
                    "columns": list(head.columns),
                })
                continue

            usable.append({
                "path": path,
                "label": label,
                "timestamp": timestamp,
                "columns": list(head.columns),
            })
        except Exception as exc:
            diagnostics.append({
                "file": str(path),
                "status": f"{type(exc).__name__}: {exc}",
                "columns": [],
            })

    return usable, diagnostics

def download_cicids_source(repo_id, local_dir, attempts=3):
    last_error = None

    for attempt in range(1, attempts + 1):
        try:
            print(
                f"Downloading/checking {repo_id} "
                f"(attempt {attempt}/{attempts})..."
            )

            local_dir = Path(local_dir)
            local_dir.mkdir(parents=True, exist_ok=True)

            # This is the same direct Hugging Face import strategy as the
            # previously working FAST v4 notebook.
            snapshot_download(
                repo_id=repo_id,
                repo_type="dataset",
                local_dir=str(local_dir),
                allow_patterns=["*.csv", "*.CSV"],
                ignore_patterns=[
                    "*.md", "*.txt", "*.json", "*.parquet",
                    "*.zip", "*.pcap", "*.pcapng"
                ],
            )

            csv_paths = discover_csv_files(local_dir)

            if not csv_paths:
                raise RuntimeError("No CSV files were downloaded.")

            usable, diagnostics = inspect_cicids_files(csv_paths)

            if len(usable) < 5:
                raise RuntimeError(
                    f"Only {len(usable)} CSVs contain Label. "
                    f"Diagnostics: {diagnostics[:5]}"
                )

            print(
                f"Accepted {repo_id}: {len(usable)} CSVs contain a valid Label column."
            )
            return local_dir, csv_paths, usable, diagnostics

        except Exception as exc:
            last_error = exc
            print(
                f"  Download/check failed: "
                f"{type(exc).__name__}: {exc}"
            )
            if attempt < attempts:
                time.sleep(2 * attempt)

    raise RuntimeError(
        f"Failed after {attempts} attempts: {last_error}"
    )

# First use an already-downloaded timestamped/local directory if it exists.
local_timestamped_dir = Path("/content/CICIDS2017_timestamped")
local_paths = discover_csv_files(local_timestamped_dir) if local_timestamped_dir.exists() else []

selected_repo = None
csv_candidates = []
schema_infos = []
repo_errors = {}

if local_paths:
    local_infos, local_diagnostics = inspect_cicids_files(local_paths)
    if len(local_infos) >= 5:
        selected_repo = "local:/content/CICIDS2017_timestamped"
        csv_candidates = local_paths
        schema_infos = local_infos
        print(
            f"Using existing CICIDS2017 files from {local_timestamped_dir}"
        )

# Otherwise download automatically.
if selected_repo is None:
    source_specs = [
        ("kumaranimesh1901/CICIDS-2017", "/content/CICIDS2017_timestamped"),
        ("bvk/CICIDS-2017", "/content/CICIDS2017_bvk"),
        ("San0160/CICIDS-2017", "/content/CICIDS2017_san0160"),
    ]

    for repo_id, target_dir in source_specs:
        try:
            _, paths, infos, diagnostics = download_cicids_source(
                repo_id,
                target_dir,
                attempts=3,
            )
            selected_repo = repo_id
            csv_candidates = paths
            schema_infos = infos
            break
        except Exception as exc:
            repo_errors[repo_id] = (
                f"{type(exc).__name__}: {exc}"
            )
            print(
                f"Rejected {repo_id} -> {repo_errors[repo_id]}"
            )

if selected_repo is None:
    raise RuntimeError(
        "No usable CICIDS2017 Hugging Face source was found. "
        "No manual dataset upload is allowed by this notebook. "
        f"Diagnostics: {repo_errors}"
    )

# Reinspect the final selected files so all later cells use one authoritative schema list.
schema_infos, candidate_diagnostics = inspect_cicids_files(csv_candidates)

if len(schema_infos) < 5:
    raise RuntimeError(
        f"Selected source has only {len(schema_infos)} usable Label CSVs. "
        f"Diagnostics: {candidate_diagnostics[:10]}"
    )

print("\nSelected CICIDS2017 source:", selected_repo)
print("CSV files found:", len(csv_candidates))
for p in csv_candidates:
    print(" -", p.name)


In [ ]:
# 5. Efficient schema discovery + schema harmonization

def normalized_columns(columns):
    return [normalize_header_name(c) for c in columns]

schema_infos = []
candidate_diagnostics = []

for path in csv_candidates:
    try:
        head = pd.read_csv(path, nrows=0, encoding="cp1252", engine="c", low_memory=False)
        label_candidate = find_exact_column(head.columns, "Label")
        timestamp_candidate = find_exact_column(head.columns, "Timestamp")

        if label_candidate is None:
            candidate_diagnostics.append({
                "file": str(path),
                "status": "Missing Label",
                "columns": list(head.columns)
            })
            continue

        schema_infos.append({
            "path": path,
            "label": label_candidate,
            "timestamp": timestamp_candidate,
            "columns": list(head.columns)
        })
    except Exception as exc:
        candidate_diagnostics.append({
            "file": str(path),
            "status": f"Header read error: {type(exc).__name__}: {exc}",
            "columns": []
        })

if not schema_infos:
    raise ValueError(
        "No CICIDS2017 CSV containing a usable Label column was loaded. "
        f"Diagnostics: {candidate_diagnostics[:10]}"
    )

print("Usable CSV files:", len(schema_infos))
print("Files with actual Timestamp:", sum(x["timestamp"] is not None for x in schema_infos))

# We do not require identical raw column names because CICIDS2017 mirrors may
# rename/remove identity fields. The model feature intersection is established
# after loading, while Label remains mandatory.


In [ ]:
# 5b. Dataset schema checks
for info in schema_infos:
    if normalize_header_name(info["label"]) != "label":
        raise ValueError(f"Unexpected Label column in {info['path']}.")

timestamp_candidates = [
    info["timestamp"] for info in schema_infos if info["timestamp"] is not None
]

if timestamp_candidates:
    print("At least one source file exposes an actual Timestamp column.")
else:
    print("No source file exposes a row-level Timestamp.")
    print("Temporal evaluation will therefore use verified CICIDS2017 capture-day/file order.")
    print("No duration/flow feature will be substituted as a timestamp.")


## 6. Temporal provenance policy

The notebook uses the strongest temporal provenance actually available in the selected
CICIDS2017 mirror:

1. **Row-level mode:** if every selected source file contains a real `Timestamp`, it is
   parsed and validated against the July 2017 capture period, then rows are sorted by it.
2. **Capture-day mode:** if the selected mirror is the standard CICIDS2017 CSV variant
   without row-level timestamps, temporal evaluation is performed by the original
   capture-day/file order (Monday → Tuesday → Wednesday → Thursday → Friday).
3. **Never:** `Flow Duration`, `Total TCP Flow Time`, packet counts, IAT values, or any
   other model feature is treated as a timestamp.

This is important because several public CICIDS2017 distributions differ in whether the
identity/timestamp fields are retained. The original eight-file organization is stable,
and the capture-day ordering is legitimate temporal provenance when row timestamps are
absent.

In [ ]:
# 6. Parse the actual Timestamp when available
def parse_cicids_timestamp(series):
    raw = (
        series.astype("string")
        .str.replace("\u00a0", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    candidates = []
    formats = [
        "%d/%m/%Y %I:%M:%S %p",
        "%m/%d/%Y %I:%M:%S %p",
        "%d/%m/%Y %H:%M:%S",
        "%m/%d/%Y %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",
        "%Y-%m-%d %H:%M:%S.%f",
        "%d/%m/%Y %I:%M:%S.%f %p",
        "%m/%d/%Y %I:%M:%S.%f %p",
    ]
    for fmt in formats:
        candidates.append(pd.to_datetime(raw, format=fmt, errors="coerce"))
    try:
        candidates.append(pd.to_datetime(raw, format="mixed", errors="coerce"))
    except Exception:
        # Compatible fallback for older pandas.
        candidates.append(pd.to_datetime(raw, errors="coerce"))

    validity = [int(x.notna().sum()) for x in candidates]
    best = candidates[int(np.argmax(validity))]
    return best, max(validity) / max(len(raw), 1)


In [ ]:
# 7. Load files one at a time, validate schema, and compact to float32 arrays

def capture_day_rank(filename):
    name = filename.casefold()
    if "monday" in name:
        return 0
    if "tuesday" in name:
        return 1
    if "wednesday" in name:
        return 2
    if "thursday" in name:
        return 3
    if "friday" in name:
        return 4
    return 99

# Sort source files by documented CICIDS2017 capture-day order, not lexicographic filename order.
schema_infos = sorted(
    schema_infos,
    key=lambda x: (capture_day_rank(x["path"].name), x["path"].name.casefold())
)

# Decide temporal provenance before loading the full feature matrix.
TIMESTAMP_VALIDITY_THRESHOLD = 0.90
TIMESTAMP_SAMPLE_ROWS = 20000
preflight_timestamp = []

has_timestamp_flags = [info["timestamp"] is not None for info in schema_infos]
ROW_TIMESTAMP_MODE = all(has_timestamp_flags)

if any(has_timestamp_flags) and not ROW_TIMESTAMP_MODE:
    print("Mixed Timestamp availability -> using capture-day temporal mode.")

if ROW_TIMESTAMP_MODE:
    print("\nPreflight: checking actual Timestamp quality...")
    for info in schema_infos:
        sample = pd.read_csv(
            info["path"], usecols=[info["timestamp"]], nrows=TIMESTAMP_SAMPLE_ROWS,
            encoding="cp1252", engine="c"
        )
        parsed_sample, sample_validity = parse_cicids_timestamp(sample[info["timestamp"]])
        valid_sample = parsed_sample.dropna()
        sample_min = valid_sample.min() if not valid_sample.empty else None
        sample_max = valid_sample.max() if not valid_sample.empty else None
        preflight_timestamp.append({
            "file": str(info["path"]), "sample_validity": float(sample_validity),
            "sample_min": None if sample_min is None else str(sample_min),
            "sample_max": None if sample_max is None else str(sample_max)
        })
        print(f"  {info['path'].name}: {sample_validity:.2%} valid")
        del sample, parsed_sample
        gc.collect()

    if any(x["sample_validity"] < TIMESTAMP_VALIDITY_THRESHOLD for x in preflight_timestamp):
        print("Actual Timestamp values are incomplete/invalid for row-level ordering.")
        print("Using whole CICIDS2017 capture-day/source-file order instead.")
        print("No timestamp will be fabricated from partial values.")
        ROW_TIMESTAMP_MODE = False
    else:
        for x in preflight_timestamp:
            for key in ("sample_min", "sample_max"):
                if x[key] is not None:
                    dt = pd.Timestamp(x[key])
                    if not (pd.Timestamp("2017-07-01") <= dt <= pd.Timestamp("2017-07-10")):
                        print("Suspicious Timestamp date range detected -> capture-day mode.")
                        ROW_TIMESTAMP_MODE = False
                        break
            if not ROW_TIMESTAMP_MODE:
                break

print("Temporal provenance selected:", "ROW_TIMESTAMP" if ROW_TIMESTAMP_MODE else "CAPTURE_DAY")

feature_parts = []
label_parts = []
timestamp_parts = []
source_file_parts = []
source_day_parts = []
loaded_files = []
file_validation = []
reference_feature_names = None

for file_id, info in enumerate(schema_infos):
    path = info["path"]
    print(f"\nReading: {path.name}")

    part = pd.read_csv(path, encoding="cp1252", low_memory=False, engine="c")

    part.columns = [
        str(c).replace("\ufeff", "").replace("\u00a0", " ").strip()
        for c in part.columns
    ]

    label_col_file = find_exact_column(part.columns, "Label")
    timestamp_col_file = find_exact_column(part.columns, "Timestamp")

    if label_col_file is None:
        raise ValueError(f"{path.name}: Label column missing after loading.")

    labels = part[label_col_file].astype("string").str.strip()
    keep = labels.notna() & (labels != "")

    parsed_ts = None
    validity = None

    if ROW_TIMESTAMP_MODE:
        parsed_ts, validity = parse_cicids_timestamp(part[timestamp_col_file])
        print(f"  Timestamp parse validity: {validity:.2%}")

        if validity < 0.90:
            raise AssertionError("Timestamp preflight selected row-level mode, but full-load validity is below threshold.")

        valid_ts = parsed_ts.dropna()
        if valid_ts.empty:
            raise ValueError(f"{path.name}: no valid timestamps remain.")

        fmin, fmax = valid_ts.min(), valid_ts.max()
        if not (
            pd.Timestamp("2017-07-01") <= fmin <= pd.Timestamp("2017-07-10")
            and pd.Timestamp("2017-07-01") <= fmax <= pd.Timestamp("2017-07-10")
        ):
            raise ValueError(
                f"{path.name}: suspicious timestamp range {fmin} -> {fmax}."
            )
        keep &= parsed_ts.notna()
    else:
        fmin = fmax = None
        print("  Row-level Timestamp: unavailable; using capture-day provenance.")

    feature_cols = [
        c for c in part.columns
        if c != label_col_file and c != timestamp_col_file
    ]

    numeric_dict = {}
    for c in feature_cols:
        values = pd.to_numeric(part[c], errors="coerce")
        if values.notna().any():
            numeric_dict[c] = values.astype(np.float32)

    if reference_feature_names is None:
        reference_feature_names = list(numeric_dict.keys())
    elif set(numeric_dict.keys()) != set(reference_feature_names):
        raise ValueError(
            f"{path.name}: numeric feature schema differs from earlier files."
        )

    X_part = pd.DataFrame(
        {c: numeric_dict[c] for c in reference_feature_names},
        index=part.index
    ).loc[keep]

    X_arr = X_part.to_numpy(dtype=np.float32, copy=True)
    X_arr[~np.isfinite(X_arr)] = np.nan

    feature_parts.append(X_arr)
    label_parts.append(labels.loc[keep].to_numpy(dtype=str))
    source_file_parts.append(np.full(int(keep.sum()), file_id, dtype=np.int16))
    source_day_parts.append(np.full(
        int(keep.sum()), capture_day_rank(path.name), dtype=np.int8
    ))

    if ROW_TIMESTAMP_MODE:
        timestamp_parts.append(
            parsed_ts.loc[keep].to_numpy(dtype="datetime64[ns]")
        )

    loaded_files.append(str(path))
    file_validation.append({
        "file": str(path),
        "rows_loaded": int(keep.sum()),
        "capture_day": capture_day_rank(path.name),
        "timestamp_validity": None if validity is None else float(validity),
        "timestamp_min": None if fmin is None else str(fmin),
        "timestamp_max": None if fmax is None else str(fmax),
    })

    print("  Retained rows:", int(keep.sum()))
    print("  Features:", X_arr.shape[1])

    del part, numeric_dict, X_part, X_arr, parsed_ts, labels
    gc.collect()

X_matrix = np.concatenate(feature_parts, axis=0).astype(np.float32, copy=False)
y_multiclass = np.concatenate(label_parts)
source_file_ids = np.concatenate(source_file_parts)
source_day_ids = np.concatenate(source_day_parts)

if ROW_TIMESTAMP_MODE:
    timestamps = np.concatenate(timestamp_parts)
else:
    timestamps = None

del feature_parts, label_parts, source_file_parts, source_day_parts
gc.collect()

timestamp_col = "Timestamp" if ROW_TIMESTAMP_MODE else None
label_col = "Label"
numeric_features = reference_feature_names

print("\nCombined shape:", X_matrix.shape)
print("Classes:", len(np.unique(y_multiclass)))
print("Temporal provenance mode:", "ROW_TIMESTAMP" if ROW_TIMESTAMP_MODE else "CAPTURE_DAY")
if ROW_TIMESTAMP_MODE:
    print("Timestamp range:", timestamps.min(), "->", timestamps.max())


In [ ]:
# 8. Compact data cleaning
# Exact duplicate removal is performed with row hashes in chunks.
# Timestamp/source provenance is included in the hash when available.

n_before = len(y_multiclass)
CHUNK = 200_000
hash_parts = []

for start in range(0, n_before, CHUNK):
    end = min(start + CHUNK, n_before)

    frame = pd.DataFrame(X_matrix[start:end], columns=numeric_features)
    frame["_label"] = y_multiclass[start:end]
    frame["_file_id"] = source_file_ids[start:end]
    frame["_day_id"] = source_day_ids[start:end]

    if ROW_TIMESTAMP_MODE:
        frame["_timestamp"] = timestamps[start:end]

    h = pd.util.hash_pandas_object(frame, index=False).to_numpy(dtype=np.uint64)
    hash_parts.append(h)

    del frame, h
    gc.collect()

row_hashes = np.concatenate(hash_parts)
del hash_parts
gc.collect()

_, unique_idx = np.unique(row_hashes, return_index=True)
unique_idx.sort()

if len(unique_idx) < n_before:
    X_matrix = X_matrix[unique_idx]
    y_multiclass = y_multiclass[unique_idx]
    source_file_ids = source_file_ids[unique_idx]
    source_day_ids = source_day_ids[unique_idx]
    if ROW_TIMESTAMP_MODE:
        timestamps = timestamps[unique_idx]

print("Duplicate rows removed:", n_before - len(unique_idx))
print("Rows after cleaning:", len(y_multiclass))
print("Classes:", len(np.unique(y_multiclass)))

del row_hashes, unique_idx
gc.collect()


In [ ]:
# 9. Final numeric feature matrix
if X_matrix.ndim != 2 or X_matrix.shape[0] != len(y_multiclass):
    raise ValueError("Feature/label row alignment is invalid.")

if source_file_ids.shape[0] != len(y_multiclass):
    raise ValueError("Source-file provenance is misaligned with features/labels.")

# Remove columns that are entirely missing.
all_missing = np.all(np.isnan(X_matrix), axis=0)
if all_missing.any():
    removed_features = [
        numeric_features[i] for i, flag in enumerate(all_missing) if flag
    ]
    X_matrix = X_matrix[:, ~all_missing]
    numeric_features = [
        c for i, c in enumerate(numeric_features) if not all_missing[i]
    ]
else:
    removed_features = []

X_matrix[~np.isfinite(X_matrix)] = np.nan

print("Numeric features retained:", len(numeric_features))
print("Completely missing features removed:", len(removed_features))
print("Raw missing cells:", int(np.isnan(X_matrix).sum()))
print("Matrix dtype:", X_matrix.dtype)

if len(numeric_features) == 0:
    raise ValueError("No usable numeric features remain.")


In [ ]:
# 10. Class summary and chronological ordering
class_counts = pd.Series(y_multiclass).value_counts()
print("Number of classes:", len(class_counts))
display(class_counts.to_frame("count").head(30))

if ROW_TIMESTAMP_MODE:
    ordered_idx = np.argsort(timestamps, kind="mergesort")
    temporal_order_description = "Actual row-level Timestamp"
else:
    # Preserve whole capture-day/file provenance. Rows inside a source file are
    # intentionally NOT given a fabricated within-file timestamp.
    ordered_idx = np.lexsort((
        np.arange(len(y_multiclass), dtype=np.int64),
        source_file_ids
    ))
    temporal_order_description = "CICIDS2017 capture-day/source-file order"

y_sorted = y_multiclass[ordered_idx]

if ROW_TIMESTAMP_MODE:
    timestamps_sorted = timestamps[ordered_idx]
    if not np.all(timestamps_sorted[:-1] <= timestamps_sorted[1:]):
        raise AssertionError("Chronological sorting failed.")
    print("\nActual timestamp range:")
    print(timestamps_sorted[0], "->", timestamps_sorted[-1])
else:
    timestamps_sorted = None
    print("\nTemporal provenance:", temporal_order_description)
    print("Source files in temporal order:")
    for fid in np.unique(source_file_ids[ordered_idx]):
        print(" ", fid, Path(loaded_files[int(fid)]).name)

print("Temporal order verified.")


## 11. Leakage-safe preprocessing

There must be **no missing values in the matrices actually passed to the estimators**.

For each split:
- fit the median imputer on training data only,
- transform train and test,
- replace any residual non-finite values defensively,
- fit the scaler on training data only for Logistic Regression.

For missingness stress, NaNs are injected **only into the test matrix**, then passed through the already-fitted train-derived imputer. This keeps the stress factor controlled and prevents leakage.


In [ ]:
# 11. Leakage-safe preprocessing + optimized model factories
from sklearn.base import clone
from scipy.stats import t

def fit_train_imputer(X_train_raw):
    """Fit the imputer exactly once for a split and reuse it across models/tasks."""
    imputer = SimpleImputer(strategy="median", keep_empty_features=True)
    Xtr = imputer.fit_transform(X_train_raw).astype(np.float32, copy=False)
    Xtr = np.nan_to_num(Xtr, nan=0.0, posinf=0.0, neginf=0.0)
    if not np.isfinite(Xtr).all():
        raise ValueError("Non-finite values remain after train-only imputation.")
    return Xtr, imputer

def transform_with_imputer(imputer, X_raw):
    out = imputer.transform(X_raw).astype(np.float32, copy=False)
    out = np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0)
    if not np.isfinite(out).all():
        raise ValueError("Non-finite values remain after train-fitted imputation.")
    return out

def make_lr(seed):
    return LogisticRegression(
        max_iter=120,
        solver="lbfgs",
        tol=1e-3,
        random_state=seed
    )

def make_rf(seed):
    return RandomForestClassifier(
        n_estimators=45 if not PAPER_MODE else 65,
        max_depth=18,
        min_samples_leaf=2,
        max_samples=0.65,
        n_jobs=N_JOBS,
        random_state=seed,
        class_weight="balanced_subsample"
    )

def make_xgb(seed, binary=False):
    if not XGB_AVAILABLE:
        raise RuntimeError("XGBoost is required for this experiment.")

    common = dict(
        n_estimators=55 if not PAPER_MODE else 85,
        max_depth=5,
        learning_rate=0.08,
        subsample=0.75,
        colsample_bytree=0.75,
        min_child_weight=2,
        max_bin=64,
        tree_method="hist",
        n_jobs=N_JOBS,
        random_state=seed,
        verbosity=0
    )

    if binary:
        return XGBClassifier(
            objective="binary:logistic",
            eval_metric="logloss",
            **common
        )

    return XGBClassifier(
        objective="multi:softmax",
        eval_metric="mlogloss",
        **common
    )

MODEL_NAMES = ["Logistic Regression", "Random Forest", "XGBoost"]
print("Optimized model factories ready.")


In [ ]:
# 12. Build reference and capture-day chronological splits using indices only
n = len(y_sorted)

class_counts_for_split = pd.Series(y_sorted).value_counts()
if class_counts_for_split.min() < 2:
    rare = class_counts_for_split[class_counts_for_split < 2].to_dict()
    raise ValueError(
        "Reference stratified split cannot be formed because some classes have "
        f"fewer than 2 rows: {rare}. The notebook will not silently delete classes."
    )

all_idx = np.arange(n, dtype=np.int64)
ref_train_idx, ref_test_idx = train_test_split(
    all_idx,
    test_size=0.20,
    random_state=SEED,
    stratify=y_sorted
)

if ROW_TIMESTAMP_MODE:
    # Row-level chronological 80/20 split.
    cut = int(0.80 * n)
    temporal_train_idx = np.arange(cut, dtype=np.int64)
    temporal_test_idx = np.arange(cut, n, dtype=np.int64)
else:
    # File/day-level chronological split. Whole capture days remain intact.
    # Monday->Thursday are training and Friday is held out when Friday exists.
    # This prevents within-day leakage and does not fabricate row timestamps.
    day_sequence = [0, 1, 2, 3, 4]
    available_days = sorted(np.unique(source_day_ids).tolist())

    if len(available_days) < 2:
        raise ValueError("At least two capture days are required for temporal evaluation.")

    test_day = available_days[-1]
    train_mask = source_day_ids[ordered_idx] < test_day
    test_mask = source_day_ids[ordered_idx] == test_day

    temporal_train_idx = np.flatnonzero(train_mask).astype(np.int64)
    temporal_test_idx = np.flatnonzero(test_mask).astype(np.int64)

    if len(temporal_train_idx) == 0 or len(temporal_test_idx) == 0:
        raise ValueError("Capture-day temporal split produced an empty partition.")

print("Reference:", len(ref_train_idx), "train /", len(ref_test_idx), "test")
print(
    "Temporal:",
    len(temporal_train_idx), "train /", len(temporal_test_idx), "test",
    "| mode =", "row-level Timestamp" if ROW_TIMESTAMP_MODE else "capture-day"
)

temporal_train_classes = set(y_sorted[temporal_train_idx])
temporal_test_classes = set(y_sorted[temporal_test_idx])
unseen_temporal_classes = sorted(temporal_test_classes - temporal_train_classes)

print("Temporal test-only classes:", len(unseen_temporal_classes))
if unseen_temporal_classes:
    print("These classes are genuinely absent from temporal training:")
    print(unseen_temporal_classes)

def take_sorted_rows(indices):
    """Materialize only requested rows; no second full feature matrix is retained."""
    indices = np.asarray(indices, dtype=np.int64)
    return X_matrix[ordered_idx[indices]]


In [ ]:
# 14. Training-size policy — full data for paper, stratified bounded data for validation
def make_training_indices(indices, y_values, max_samples, seed):
    """
    PAPER_MODE: return the complete training partition.

    QUICK_VALIDATION: use a reproducible stratified sample so the fast run
    preserves approximate class proportions. This is only a pipeline check,
    never the final paper protocol.
    """
    indices = np.asarray(indices, dtype=np.int64)

    if PAPER_MODE or len(indices) <= max_samples:
        return indices.copy()

    y_arr = np.asarray(y_values)
    labels = y_arr[indices]
    unique, counts = np.unique(labels, return_counts=True)

    if not FAST_SAMPLING_STRATIFIED:
        rng = np.random.default_rng(seed)
        return np.sort(rng.choice(indices, size=max_samples, replace=False))

    rng = np.random.default_rng(seed)
    # Largest-remainder allocation so the selected sample approximately follows
    # the original class distribution while guaranteeing at least one example
    # per class when the budget permits.
    if max_samples < len(unique):
        raise ValueError("FAST_TRAIN_SAMPLES must be >= number of classes.")

    target = max_samples * counts / counts.sum()
    quotas = np.floor(target).astype(int)
    quotas = np.maximum(quotas, 1)

    # Reduce if minimum-one allocation exceeded the budget.
    while quotas.sum() > max_samples:
        reducible = np.where(quotas > 1)[0]
        if len(reducible) == 0:
            break
        # Remove from the class with the largest quota relative to its target.
        j = reducible[np.argmax(quotas[reducible] - target[reducible])]
        quotas[j] -= 1

    # Distribute remaining slots by largest fractional remainder.
    remaining = max_samples - quotas.sum()
    fractions = target - np.floor(target)
    order = np.argsort(-fractions)
    for j in order:
        if remaining <= 0:
            break
        if quotas[j] < counts[j]:
            quotas[j] += 1
            remaining -= 1

    selected_parts = []
    for cls, quota in zip(unique, quotas):
        cls_idx = indices[labels == cls]
        quota = min(int(quota), len(cls_idx))
        selected_parts.append(rng.choice(cls_idx, size=quota, replace=False))

    selected = np.concatenate(selected_parts).astype(np.int64)
    rng.shuffle(selected)
    return selected

ref_fit_idx = make_training_indices(
    ref_train_idx, y_sorted, FAST_TRAIN_SAMPLES, SEED
)
temp_fit_idx = make_training_indices(
    temporal_train_idx, y_sorted, FAST_TRAIN_SAMPLES, SEED
)

print("Reference fit rows:", len(ref_fit_idx))
print("Temporal fit rows:", len(temp_fit_idx))
print("Paper mode:", PAPER_MODE)
print("Fast sampling:", "stratified" if FAST_SAMPLING_STRATIFIED else "uniform")


In [ ]:
# 14. Evaluation metrics
def calculate_fpr_from_cm(cm):
    cm = np.asarray(cm)
    total = cm.sum()
    fprs = []
    for i in range(cm.shape[0]):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = total - tp - fn - fp
        denom = fp + tn
        fprs.append(fp / denom if denom else 0.0)
    return float(np.mean(fprs))

def evaluate_predictions(y_true, y_pred, labels):
    labels = list(labels)
    acc = accuracy_score(y_true, y_pred)
    p, r, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, labels=labels, zero_division=0
    )
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    return {
        "Accuracy": float(acc),
        "Macro Precision": float(np.mean(p)),
        "Macro Recall": float(np.mean(r)),
        "Macro F1": float(np.mean(f1)),
        "Weighted F1": float(
            precision_recall_fscore_support(
                y_true, y_pred, labels=labels, average="weighted", zero_division=0
            )[2]
        ),
        "FPR": calculate_fpr_from_cm(cm),
        "Confusion Matrix": cm,
        "Per-class Recall": dict(zip(labels, r))
    }


In [ ]:
# 15. Memory-efficient feature missingness stress
def inject_feature_missingness(X, rate, seed):
    """
    Inject independent feature-value missingness into the evaluation matrix.
    The original matrix is never modified.

    A column-wise 1-D mask avoids allocating a full rows x features boolean matrix.
    """
    if rate <= 0:
        return X

    rng = np.random.default_rng(seed)
    X_stress = X.copy()

    for j in range(X_stress.shape[1]):
        mask = rng.random(X_stress.shape[0]) < rate
        X_stress[mask, j] = np.nan

    return X_stress

def apply_fitted_imputer(imputer, X):
    return transform_with_imputer(imputer, X)


In [ ]:
# 16. Fit model using a split-level, already-fitted imputer
def fit_model_for_split(
    model_name, X_train_imputed, imputer, y_train, seed, binary=False
):
    if model_name == "Logistic Regression":
        model = make_lr(seed)
    elif model_name == "Random Forest":
        model = make_rf(seed)
    elif model_name == "XGBoost":
        model = make_xgb(seed, binary=binary)
    else:
        raise ValueError("Unknown model: " + model_name)

    X_train = X_train_imputed
    scaler = None

    if model_name == "Logistic Regression":
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train).astype(np.float32, copy=False)

    label_classes = None
    if model_name == "XGBoost" and not binary:
        label_classes, y_encoded = np.unique(
            np.asarray(y_train).astype(str), return_inverse=True
        )
        model.fit(X_train, y_encoded)
    else:
        model.fit(X_train, y_train)

    return {
        "model": model,
        "imputer": imputer,
        "scaler": scaler,
        "label_classes": label_classes,
        "binary": binary,
        "model_name": model_name,
    }

def transform_test(fitted, X_test_raw):
    X_test = apply_fitted_imputer(fitted["imputer"], X_test_raw)

    if fitted["scaler"] is not None:
        X_test = fitted["scaler"].transform(X_test).astype(np.float32, copy=False)

    if not np.isfinite(X_test).all():
        raise ValueError("Non-finite values remain in the estimator input.")

    return X_test


In [ ]:
# 17. Safe multiclass/binary prediction wrapper
def prepare_binary_labels(y):
    arr = np.asarray(y)
    if np.issubdtype(arr.dtype, np.number):
        unique = set(np.unique(arr).tolist())
        if unique.issubset({0, 1}):
            return arr.astype(np.int32)
    return np.asarray(
        [0 if str(v).strip().casefold() == "benign" else 1 for v in arr],
        dtype=np.int32
    )

def evaluate_fitted(
    fitted, X_test_raw, y_test, labels, condition, model_name, binary=False,
    missingness=0.0, seed=42
):
    X_eval_raw = X_test_raw
    if missingness > 0:
        X_eval_raw = inject_feature_missingness(X_test_raw, missingness, seed)

    X_eval = transform_test(fitted, X_eval_raw)
    raw_pred = fitted["model"].predict(X_eval)

    if binary:
        y_true_eval = prepare_binary_labels(y_test)
        labels_eval = [0, 1]
        y_pred = np.asarray(raw_pred).astype(int)
    else:
        y_true_eval = np.asarray(y_test).astype(str)
        labels_eval = list(labels)

        if model_name == "XGBoost" and fitted.get("label_classes") is not None:
            encoded_pred = np.asarray(raw_pred).astype(int)
            known_classes = np.asarray(fitted["label_classes"]).astype(str)
            y_pred = known_classes[encoded_pred]
        else:
            y_pred = np.asarray(raw_pred).astype(str)

    metrics = evaluate_predictions(y_true_eval, y_pred, labels_eval)
    metrics.update({
        "Task": "binary" if binary else "multiclass",
        "Model": model_name,
        "Condition": condition,
        "Missingness": float(missingness),
        "Seed": int(seed),
    })
    return metrics


## 18. Experiment runner

The runner explicitly separates:
- reference vs temporal training data,
- binary vs multiclass labels,
- normal vs stressed test inputs.

The fitted model is reused for normal, 10% missingness, and 20% missingness evaluation. This is important: missingness is a **test-time stress factor**, not a second training protocol.


In [ ]:
# 18. Experiment runner — fit once per model/split; reuse split-level preprocessing
CONDITION_SPECS = [
    ("Reference", 0.0),
    ("Reference+10% Missingness", 0.10),
    ("Reference+20% Missingness", 0.20),
    ("Temporal", 0.0),
    ("Temporal+10% Missingness", 0.10),
    ("Temporal+20% Missingness", 0.20),
]

def run_one_seed(seed):
    np.random.seed(seed)
    random.seed(seed)

    y_all = y_sorted

    # Reference split changes by seed; temporal split is fixed by provenance.
    all_idx = np.arange(len(y_all), dtype=np.int64)
    ref_train_idx_s, ref_test_idx_s = train_test_split(
        all_idx,
        test_size=0.20,
        random_state=seed,
        stratify=y_all
    )

    temp_train_idx_s = temporal_train_idx
    temp_test_idx_s = temporal_test_idx

    ref_fit_idx_s = make_training_indices(
        ref_train_idx_s, y_all, FAST_TRAIN_SAMPLES, seed
    )
    temp_fit_idx_s = make_training_indices(
        temp_train_idx_s, y_all, FAST_TRAIN_SAMPLES, seed
    )

    split_indices = {
        "Reference": (ref_fit_idx_s, ref_test_idx_s),
        "Temporal": (temp_fit_idx_s, temp_test_idx_s),
    }

    # Materialize each split once and fit the median imputer once.
    prepared = {}
    for split_name, (fit_idx, test_idx) in split_indices.items():
        X_train_raw = take_sorted_rows(fit_idx)
        X_test_raw = take_sorted_rows(test_idx)
        X_train_imputed, imputer = fit_train_imputer(X_train_raw)

        prepared[split_name] = {
            "fit_idx": fit_idx,
            "test_idx": test_idx,
            "X_train_imputed": X_train_imputed,
            "X_test_raw": X_test_raw,
            "imputer": imputer,
        }
        del X_train_raw
        gc.collect()

    rows = []
    reports = {}

    for task in ["multiclass", "binary"]:
        binary = task == "binary"
        y_task = prepare_binary_labels(y_all) if binary else y_all
        labels = [0, 1] if binary else sorted(np.unique(y_task))

        for split_name in ["Reference", "Temporal"]:
            p = prepared[split_name]
            fit_idx = p["fit_idx"]
            test_idx = p["test_idx"]
            y_train = y_task[fit_idx]
            y_test = y_task[test_idx]

            print(
                f"[Seed {seed}] {task} | {split_name} | "
                f"train={len(fit_idx):,} test={len(test_idx):,}"
            )

            for model_name in MODEL_NAMES:
                print(f"  -> {model_name}")

                # One fit per model/split/task. All three stress levels reuse it.
                fitted = fit_model_for_split(
                    model_name=model_name,
                    X_train_imputed=p["X_train_imputed"],
                    imputer=p["imputer"],
                    y_train=y_train,
                    seed=seed,
                    binary=binary,
                )

                for condition, missingness in CONDITION_SPECS:
                    if condition.startswith(split_name):
                        result = evaluate_fitted(
                            fitted=fitted,
                            X_test_raw=p["X_test_raw"],
                            y_test=y_test,
                            labels=labels,
                            condition=condition,
                            model_name=model_name,
                            binary=binary,
                            missingness=missingness,
                            seed=seed,
                        )
                        rows.append(result)

                        # Keep every multiclass condition for attack-level analysis.
                        if task == "multiclass":
                            reports[(task, condition, model_name)] = result

                del fitted
                gc.collect()

    del prepared
    gc.collect()

    return pd.DataFrame(rows), reports


In [ ]:
# 19. Run the primary experiment
results_all = pd.DataFrame()
reports_all = {}

if RUN_EXPERIMENTS:
    results_all, reports_all = run_one_seed(SEED)
    print("\nPrimary experiment complete.")

    display(
        results_all[
            ["Task", "Model", "Condition", "Missingness", "Accuracy",
             "Macro Precision", "Macro Recall", "Macro F1", "Weighted F1", "FPR", "Seed"]
        ].sort_values(["Task", "Model", "Condition"])
    )
else:
    print("RUN_EXPERIMENTS=False: experiment execution skipped.")


In [ ]:
# 20. Degradation analysis
# `degradation_f1`, `degradation_recall`, and `degradation_fpr` are finalized
# after repeated-seed aggregation in Cell 23. This cell provides a quick
# seed-42 view for validation runs.
def degradation_table(results, metric="Macro F1"):
    base = (
        results[results["Condition"] == "Reference"]
        [["Task", "Model", metric]]
        .rename(columns={metric: "Reference"})
    )
    merged = results.merge(base, on=["Task", "Model"], how="left")

    if metric == "FPR":
        merged["Degradation"] = (
            (merged[metric] - merged["Reference"]) /
            merged["Reference"].replace(0, np.nan)
        )
    else:
        merged["Degradation"] = (
            (merged["Reference"] - merged[metric]) /
            merged["Reference"].replace(0, np.nan)
        )

    return merged[
        ["Task", "Model", "Condition", metric, "Reference", "Degradation", "Seed"]
    ].copy()

if not results_all.empty:
    primary_degradation_f1 = degradation_table(results_all, "Macro F1")
    primary_degradation_recall = degradation_table(results_all, "Macro Recall")
    primary_degradation_fpr = degradation_table(results_all, "FPR")
    print("Seed-42 Macro-F1 degradation preview:")
    display(primary_degradation_f1)
else:
    primary_degradation_f1 = pd.DataFrame()
    primary_degradation_recall = pd.DataFrame()
    primary_degradation_fpr = pd.DataFrame()


In [ ]:
# 21. Interaction analysis
# Final interaction results are computed from the same final condition summary
# used for the paper tables (five-seed mean in PAPER_MODE; seed-42 in validation).
if not final_interaction_results.empty:
    display(final_interaction_results)
else:
    print("Interaction table will populate after the repeated/primary experiment.")


In [ ]:
# 22. Attack-level recall analysis across ALL reliability conditions
def attack_recall_table(reports):
    rows = []

    for (task, condition, model_name), result in reports.items():
        if task != "multiclass":
            continue

        for cls, recall in result["Per-class Recall"].items():
            rows.append({
                "Task": task,
                "Condition": condition,
                "Model": model_name,
                "Class": cls,
                "Recall": float(recall),
                "Seed": int(result["Seed"]),
            })

    return pd.DataFrame(rows)

def attack_recall_condition_degradation(attack_df):
    if attack_df.empty:
        return pd.DataFrame()

    base = (
        attack_df[attack_df["Condition"] == "Reference"]
        [["Model", "Class", "Recall"]]
        .rename(columns={"Recall": "Reference Recall"})
    )

    out = attack_df.merge(base, on=["Model", "Class"], how="left")
    out["Recall Degradation"] = (
        out["Reference Recall"] - out["Recall"]
    )
    out["Relative Recall Degradation"] = (
        out["Recall Degradation"] /
        out["Reference Recall"].replace(0, np.nan)
    )
    return out

attack_recall = attack_recall_table(reports_all) if reports_all else pd.DataFrame()
attack_recall_degradation = (
    attack_recall_condition_degradation(attack_recall)
    if not attack_recall.empty else pd.DataFrame()
)

if not attack_recall_degradation.empty:
    display(
        attack_recall_degradation[
            attack_recall_degradation["Condition"] != "Reference"
        ].sort_values(
            ["Model", "Condition", "Recall Degradation"],
            ascending=[True, True, False]
        ).head(60)
    )
else:
    print("Attack-level condition table will populate after the main experiment.")


## 23. Repeated seeds

Paper-mode reproducibility uses:

`42, 52, 62, 72, 82`

Seed 42 is already produced by the primary experiment. To avoid training the same
models twice, the repeated-seed stage runs only the remaining four seeds and then
combines them with the primary seed-42 results.

In [ ]:
# 23. Repeated seeds + statistically correct final aggregation
def run_repeated_seeds():
    all_runs = [results_all.copy()] if not results_all.empty else []
    all_report_runs = [reports_all.copy()] if reports_all else []

    # Seed 42 is already produced by the primary run; avoid duplicate fitting.
    for s in SEEDS[1:]:
        print(f"\n========== SEED {s} ==========")
        run_df, run_reports = run_one_seed(s)
        all_runs.append(run_df)
        all_report_runs.append(run_reports)

    if not all_runs:
        return pd.DataFrame(), {}

    combined_reports = {}
    for report_dict in all_report_runs:
        # Include seed in the key so repeated conditions remain distinct.
        for (task, condition, model_name), result in report_dict.items():
            key = (task, condition, model_name, int(result["Seed"]))
            combined_reports[key] = result

    return pd.concat(all_runs, ignore_index=True), combined_reports

def mean_ci(series, confidence=CI_LEVEL):
    values = pd.to_numeric(series, errors="coerce").dropna().to_numpy(dtype=float)
    n = len(values)
    if n == 0:
        return np.nan, np.nan, np.nan, np.nan, 0
    mean = float(np.mean(values))
    if n == 1:
        return mean, np.nan, np.nan, np.nan, 1
    std = float(np.std(values, ddof=1))
    critical = float(t.ppf((1 + confidence) / 2, df=n - 1))
    half_width = critical * std / np.sqrt(n)
    return mean, std, mean - half_width, mean + half_width, n

def summarize_repeated_results(results):
    rows = []
    for keys, g in results.groupby(["Task", "Model", "Condition"]):
        task, model, condition = keys
        mf1, sf1, lo_f1, hi_f1, n = mean_ci(g["Macro F1"])
        mr, sr, lo_r, hi_r, _ = mean_ci(g["Macro Recall"])
        fpr, sfpr, lo_fpr, hi_fpr, _ = mean_ci(g["FPR"])

        rows.append({
            "Task": task,
            "Model": model,
            "Condition": condition,
            "Mean_Macro_F1": mf1,
            "Std_Macro_F1": sf1,
            "CI95_Macro_F1_Low": lo_f1,
            "CI95_Macro_F1_High": hi_f1,
            "Mean_Macro_Recall": mr,
            "Std_Macro_Recall": sr,
            "CI95_Macro_Recall_Low": lo_r,
            "CI95_Macro_Recall_High": hi_r,
            "Mean_FPR": fpr,
            "Std_FPR": sfpr,
            "CI95_FPR_Low": lo_fpr,
            "CI95_FPR_High": hi_fpr,
            "N": n,
        })
    return pd.DataFrame(rows)

if FINAL_MODE and RUN_EXPERIMENTS:
    repeated_results, repeated_reports = run_repeated_seeds()
else:
    repeated_results = results_all.copy() if not results_all.empty else pd.DataFrame()
    repeated_reports = {}

repeated_summary = (
    summarize_repeated_results(repeated_results)
    if not repeated_results.empty else pd.DataFrame()
)

# Final paper tables are based on the five-seed means in PAPER_MODE.
# Validation mode uses its single seed only so the notebook remains fast.
if FINAL_MODE and not repeated_results.empty:
    final_condition_metrics = (
        repeated_results
        .groupby(["Task", "Model", "Condition"], as_index=False)
        .agg(
            Accuracy=("Accuracy", "mean"),
            Macro_Precision=("Macro Precision", "mean"),
            Macro_Recall=("Macro Recall", "mean"),
            Macro_F1=("Macro F1", "mean"),
            Weighted_F1=("Weighted F1", "mean"),
            FPR=("FPR", "mean"),
        )
    )
else:
    final_condition_metrics = results_all.rename(
        columns={
            "Macro Precision": "Macro_Precision",
            "Macro Recall": "Macro_Recall",
            "Macro F1": "Macro_F1",
            "Weighted F1": "Weighted_F1",
        }
    )[[
        "Task", "Model", "Condition", "Accuracy",
        "Macro_Precision", "Macro_Recall", "Macro_F1",
        "Weighted_F1", "FPR"
    ]].copy()

def degradation_from_summary(summary, metric_col):
    base = (
        summary[summary["Condition"] == "Reference"]
        [["Task", "Model", metric_col]]
        .rename(columns={metric_col: "Reference"})
    )
    merged = summary.merge(base, on=["Task", "Model"], how="left")

    if metric_col == "FPR":
        merged["Degradation"] = (
            (merged[metric_col] - merged["Reference"]) /
            merged["Reference"].replace(0, np.nan)
        )
    else:
        merged["Degradation"] = (
            (merged["Reference"] - merged[metric_col]) /
            merged["Reference"].replace(0, np.nan)
        )

    return merged[
        ["Task", "Model", "Condition", metric_col, "Reference", "Degradation"]
    ].copy()

final_degradation_f1 = degradation_from_summary(final_condition_metrics, "Macro_F1")
final_degradation_recall = degradation_from_summary(final_condition_metrics, "Macro_Recall")
final_degradation_fpr = degradation_from_summary(final_condition_metrics, "FPR")

def interaction_from_degradation(deg):
    out = []
    for (task, model), g in deg.groupby(["Task", "Model"]):
        lookup = dict(zip(g["Condition"], g["Degradation"]))

        d_temp = lookup.get("Temporal")
        for rate in ("10%", "20%"):
            d_m = lookup.get(f"Reference+{rate} Missingness")
            d_c = lookup.get(f"Temporal+{rate} Missingness")
            if d_temp is not None and d_m is not None and d_c is not None:
                out.append({
                    "Task": task,
                    "Model": model,
                    "Missingness": rate,
                    "Temporal Degradation": d_temp,
                    "Missingness Degradation": d_m,
                    "Combined Degradation": d_c,
                    "Interaction": d_c - (d_temp + d_m),
                })
    return pd.DataFrame(out)

final_interaction_results = interaction_from_degradation(final_degradation_f1)

# Backward-compatible names used by export/figure cells.
degradation_f1 = final_degradation_f1
degradation_recall = final_degradation_recall
degradation_fpr = final_degradation_fpr
interaction_results = final_interaction_results


# Attack-level reliability analysis is also aggregated across all seeds in paper mode.
def attack_table_from_reports(report_dict, aggregate=True):
    rows = []
    for key, result in report_dict.items():
        if len(key) == 4:
            task, condition, model_name, seed = key
        else:
            task, condition, model_name = key
            seed = result["Seed"]
        if task != "multiclass":
            continue
        for cls, recall in result["Per-class Recall"].items():
            rows.append({
                "Task": task,
                "Condition": condition,
                "Model": model_name,
                "Class": cls,
                "Recall": float(recall),
                "Seed": int(seed),
            })
    df = pd.DataFrame(rows)
    if df.empty:
        return df
    if aggregate:
        return (
            df.groupby(["Task", "Condition", "Model", "Class"], as_index=False)
            .agg(
                Mean_Recall=("Recall", "mean"),
                Std_Recall=("Recall", "std"),
                N=("Recall", "count")
            )
        )
    return df

if FINAL_MODE and repeated_reports:
    attack_recall_final = attack_table_from_reports(repeated_reports, aggregate=True)
else:
    attack_recall_final = attack_table_from_reports(
        reports_all, aggregate=True
    ) if reports_all else pd.DataFrame()

if not attack_recall_final.empty:
    ref = (
        attack_recall_final[attack_recall_final["Condition"] == "Reference"]
        [["Model", "Class", "Mean_Recall"]]
        .rename(columns={"Mean_Recall": "Reference_Recall"})
    )
    attack_recall_final_degradation = attack_recall_final.merge(
        ref, on=["Model", "Class"], how="left"
    )
    attack_recall_final_degradation["Recall_Degradation"] = (
        attack_recall_final_degradation["Reference_Recall"] -
        attack_recall_final_degradation["Mean_Recall"]
    )
    attack_recall_final_degradation["Relative_Recall_Degradation"] = (
        attack_recall_final_degradation["Recall_Degradation"] /
        attack_recall_final_degradation["Reference_Recall"].replace(0, np.nan)
    )
else:
    attack_recall_final_degradation = pd.DataFrame()

# Use the all-condition final attack tables for downstream figures/exports.
attack_recall = attack_recall_final
attack_recall_degradation = attack_recall_final_degradation

display(repeated_summary if not repeated_summary.empty else final_condition_metrics)


In [ ]:
# 24. Final-paper figures — compact, reproducible, and based on final summaries
import matplotlib.pyplot as plt

if not final_condition_metrics.empty:
    # Figure 1 — Reference vs temporal Macro-F1
    plot_df = final_condition_metrics[
        (final_condition_metrics["Task"] == "multiclass") &
        (final_condition_metrics["Condition"].isin(["Reference", "Temporal"]))
    ].copy()

    fig, ax = plt.subplots(figsize=(9, 5))
    for model in MODEL_NAMES:
        g = plot_df[plot_df["Model"] == model]
        ax.plot(g["Condition"], g["Macro_F1"], marker="o", label=model)
    ax.set_title("Reference vs Chronological Macro-F1")
    ax.set_ylabel("Macro-F1")
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figure1_reference_vs_temporal_macro_f1.png", dpi=180)
    plt.show()
    plt.close(fig)

    # Figure 2 — degradation across all reliability conditions
    d = degradation_f1[
        (degradation_f1["Task"] == "multiclass") &
        (~degradation_f1["Condition"].eq("Reference"))
    ]
    fig, ax = plt.subplots(figsize=(12, 5))
    for model in MODEL_NAMES:
        g = d[d["Model"] == model]
        ax.plot(g["Condition"], g["Degradation"], marker="o", label=model)
    ax.axhline(0, linewidth=1)
    ax.set_title("Macro-F1 Degradation by Reliability Condition")
    ax.set_ylabel("Relative degradation")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "figure2_degradation.png", dpi=180)
    plt.show()
    plt.close(fig)

    # Figure 3 — temporal × missingness interaction
    if not final_interaction_results.empty:
        g = final_interaction_results[final_interaction_results["Task"] == "multiclass"]
        fig, ax = plt.subplots(figsize=(9, 5))
        for model in MODEL_NAMES:
            z = g[g["Model"] == model]
            ax.plot(z["Missingness"], z["Interaction"], marker="o", label=model)
        ax.axhline(0, linewidth=1)
        ax.set_title("Temporal × Missingness Interaction")
        ax.set_ylabel("Interaction effect")
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "figure3_interaction.png", dpi=180)
        plt.show()
        plt.close(fig)

    # Figure 4 — worst attack-level recall degradation by condition.
    if not attack_recall_degradation.empty:
        g = attack_recall_degradation[
            attack_recall_degradation["Condition"] != "Reference"
        ].copy()
        fig, ax = plt.subplots(figsize=(12, 6))
        # Show the ten largest degradation values across all model/condition/class combinations.
        z = (
            g.sort_values("Recall_Degradation", ascending=False)
            .head(10)
        )
        labels = (
            z["Model"].astype(str) + " | " +
            z["Condition"].astype(str) + " | " +
            z["Class"].astype(str)
        )
        ax.bar(np.arange(len(z)), z["Recall_Degradation"])
        ax.set_xticks(np.arange(len(z)))
        ax.set_xticklabels(labels, rotation=75, ha="right")
        ax.set_title("Largest Attack-Class Recall Degradation")
        ax.set_ylabel("Reference Recall − Condition Recall")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "figure4_attack_recall_degradation.png", dpi=180)
        plt.show()
        plt.close(fig)

    # Figure 5 — representative confusion matrix from seed-42 reference run.
    cm_candidates = results_all[
        (results_all["Task"] == "multiclass") &
        (results_all["Condition"] == "Reference")
    ].sort_values("Macro F1", ascending=False)

    if not cm_candidates.empty:
        best_row = cm_candidates.iloc[0]
        cm = np.asarray(best_row["Confusion Matrix"])
        fig, ax = plt.subplots(figsize=(10, 8))
        im = ax.imshow(cm, aspect="auto")
        ax.set_title(f"Reference Confusion Matrix — {best_row['Model']}")
        ax.set_xlabel("Predicted class index")
        ax.set_ylabel("True class index")
        fig.colorbar(im, ax=ax, label="Count")
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "figure5_representative_confusion_matrix.png", dpi=180)
        plt.show()
        plt.close(fig)

    # Figure 6 — five-seed Macro-F1 variability with t-based 95% CI.
    if not repeated_summary.empty and repeated_summary["N"].max() > 1:
        g = repeated_summary[
            (repeated_summary["Task"] == "multiclass") &
            (repeated_summary["Condition"].isin(["Reference", "Temporal"]))
        ]

        fig, ax = plt.subplots(figsize=(10, 5))
        for model in MODEL_NAMES:
            z = g[g["Model"] == model]
            x = np.arange(len(z))
            yerr_low = z["Mean_Macro_F1"] - z["CI95_Macro_F1_Low"]
            yerr_high = z["CI95_Macro_F1_High"] - z["Mean_Macro_F1"]
            ax.errorbar(
                x, z["Mean_Macro_F1"],
                yerr=[yerr_low, yerr_high],
                marker="o", capsize=4, label=model
            )

        conditions = sorted(g["Condition"].unique())
        ax.set_xticks(range(len(conditions)))
        ax.set_xticklabels(conditions)
        ax.set_title("Repeated-Seed Macro-F1 with 95% t-Confidence Intervals")
        ax.set_ylabel("Mean Macro-F1")
        ax.grid(alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "figure6_repeated_seed_variability.png", dpi=180)
        plt.show()
        plt.close(fig)
else:
    print("Figures skipped because no experiment results are available.")


In [ ]:
# 25. Automated scientific + reproducibility validation
EXPECTED_CONDITIONS = {
    "Reference",
    "Reference+10% Missingness",
    "Reference+20% Missingness",
    "Temporal",
    "Temporal+10% Missingness",
    "Temporal+20% Missingness",
}

validation = {
    "dataset_source": selected_repo,
    "loaded_files": loaded_files,
    "file_validation": file_validation,
    "temporal_provenance_mode": (
        "row_level_timestamp" if ROW_TIMESTAMP_MODE else "capture_day_file_order"
    ),
    "timestamp_column": timestamp_col,
    "timestamp_parse_min": (
        None if not ROW_TIMESTAMP_MODE
        else float(min(x["timestamp_validity"] for x in file_validation))
    ),
    "timestamp_min": (
        None if not ROW_TIMESTAMP_MODE else str(timestamps_sorted.min())
    ),
    "timestamp_max": (
        None if not ROW_TIMESTAMP_MODE else str(timestamps_sorted.max())
    ),
    "rows_final": int(len(y_multiclass)),
    "numeric_features": int(len(numeric_features)),
    "classes": int(len(np.unique(y_multiclass))),
    "raw_feature_nan_cells": int(np.isnan(X_matrix).sum()),
    "raw_feature_inf_cells": int(np.isinf(X_matrix).sum()),
    "experiments_completed": bool(not results_all.empty),
    "quick_validation": bool(QUICK_VALIDATION),
    "paper_mode": bool(PAPER_MODE),
    "fast_train_samples": int(FAST_TRAIN_SAMPLES),
    "seeds": list(SEEDS),
    "missingness_levels": list(MISSINGNESS_LEVELS),
    "condition_count_expected": len(EXPECTED_CONDITIONS),
}

if ROW_TIMESTAMP_MODE:
    if validation["timestamp_parse_min"] < 0.90:
        raise AssertionError("Actual Timestamp validation failed.")

    ts_min = pd.Timestamp(timestamps_sorted.min())
    ts_max = pd.Timestamp(timestamps_sorted.max())
    if not (
        pd.Timestamp("2017-07-01") <= ts_min <= pd.Timestamp("2017-07-10")
        and pd.Timestamp("2017-07-01") <= ts_max <= pd.Timestamp("2017-07-10")
    ):
        raise AssertionError(
            f"Global Timestamp range is not a valid CICIDS2017 timeline: {ts_min} -> {ts_max}"
        )

    # Explicitly prove that a forbidden flow-duration feature was not selected.
    if timestamp_col is not None and normalize_header_name(timestamp_col) in FORBIDDEN_TIMESTAMP_FEATURES:
        raise AssertionError("A forbidden flow feature was selected as Timestamp.")
else:
    if len(np.unique(source_day_ids)) < 2:
        raise AssertionError("Capture-day temporal mode requires at least two source days.")

if validation["classes"] < 2:
    raise AssertionError("At least two classes are required.")
if validation["numeric_features"] < 1:
    raise AssertionError("No numeric features remain.")

# Binary-label sanity checks.
assert np.array_equal(
    prepare_binary_labels(np.array([0, 1, 1, 0])),
    np.array([0, 1, 1, 0])
)
assert np.array_equal(
    prepare_binary_labels(np.array(["BENIGN", "Attack", "BENIGN"])),
    np.array([0, 1, 0])
)

# Raw non-finite values are allowed because the train-only imputer handles them,
# but no estimator input may contain non-finite values.
if not np.isfinite(np.nan_to_num(
    X_matrix, nan=0.0, posinf=0.0, neginf=0.0
)).all():
    raise AssertionError("Feature matrix contains an unhandled numeric value.")

if results_all.empty and RUN_EXPERIMENTS:
    raise AssertionError("Experiments were requested but produced no results.")

if not results_all.empty:
    observed = set(results_all["Condition"].unique())
    missing_conditions = EXPECTED_CONDITIONS - observed
    if missing_conditions:
        raise AssertionError(f"Missing reliability conditions: {sorted(missing_conditions)}")

    required_tasks = {"binary", "multiclass"}
    if set(results_all["Task"].unique()) != required_tasks:
        raise AssertionError("Both binary and multiclass experiments are required.")

    required_models = set(MODEL_NAMES)
    if set(results_all["Model"].unique()) != required_models:
        raise AssertionError("All three required models are not present.")

    # Exactly one row per task/model/condition/seed is expected.
    dupes = results_all.duplicated(
        ["Task", "Model", "Condition", "Seed"]
    ).sum()
    if dupes:
        raise AssertionError(f"Duplicate experiment rows detected: {dupes}")

    if PAPER_MODE:
        if len(repeated_results["Seed"].unique()) != len(SEEDS):
            raise AssertionError("Paper mode did not complete all required seeds.")
        if not set(SEEDS).issubset(set(repeated_results["Seed"].unique())):
            raise AssertionError("Paper-mode seed coverage is incomplete.")

        # Final results must use the complete training partitions.
        if len(ref_fit_idx) != len(ref_train_idx):
            raise AssertionError("Paper mode is not using the full reference training partition.")
        if len(temp_fit_idx) != len(temporal_train_idx):
            raise AssertionError("Paper mode is not using the full temporal training partition.")

    # Interaction completeness for both missingness levels.
    if final_interaction_results.empty:
        raise AssertionError("Interaction analysis is empty.")
    if PAPER_MODE:
        required_interactions = {
            (task, model, rate)
            for task in ["binary", "multiclass"]
            for model in MODEL_NAMES
            for rate in ["10%", "20%"]
        }
        observed_interactions = set(
            map(tuple, final_interaction_results[["Task", "Model", "Missingness"]].to_numpy())
        )
        if not required_interactions.issubset(observed_interactions):
            raise AssertionError("Paper-mode interaction coverage is incomplete.")

    # Multiclass attack-level coverage must include every model and non-reference condition.
    if attack_recall_degradation.empty:
        raise AssertionError("Attack-level reliability analysis is empty.")
    attack_conditions = set(
        attack_recall_degradation["Condition"].unique()
    ) - {"Reference"}
    if PAPER_MODE and attack_conditions != EXPECTED_CONDITIONS - {"Reference"}:
        raise AssertionError("Attack-level analysis does not cover all stress conditions.")

validation["validation_status"] = "PASS"
print(json.dumps(validation, indent=2))
print("Validation status: PASS")
print("✓ Actual timestamp or documented capture-day provenance only")
print("✓ No duration/flow feature can be used as a timestamp")
print("✓ Train-only imputation and LR scaling")
print("✓ Test-only 10%/20% feature-value missingness")
print("✓ Binary + multiclass")
print("✓ Reference + temporal + combined conditions")
print("✓ Degradation + interaction analysis")
print("✓ Attack-level analysis across all conditions")
print("✓ Full-data paper mode + five reproducible seeds")


In [ ]:
# 26. Export results, validation, and reproducibility metadata
if not results_all.empty:
    results_all.to_csv(OUTPUT_DIR / "all_results_seed42.csv", index=False)
    final_condition_metrics.to_csv(OUTPUT_DIR / "final_condition_metrics.csv", index=False)
    degradation_f1.to_csv(OUTPUT_DIR / "degradation_macro_f1.csv", index=False)
    degradation_recall.to_csv(OUTPUT_DIR / "degradation_macro_recall.csv", index=False)
    degradation_fpr.to_csv(OUTPUT_DIR / "degradation_fpr.csv", index=False)
    final_interaction_results.to_csv(OUTPUT_DIR / "interaction_results.csv", index=False)
    attack_recall.to_csv(OUTPUT_DIR / "attack_level_recall.csv", index=False)
    attack_recall_degradation.to_csv(
        OUTPUT_DIR / "attack_recall_degradation.csv", index=False
    )

if not repeated_results.empty:
    repeated_results.to_csv(OUTPUT_DIR / "repeated_seed_results.csv", index=False)

if not repeated_summary.empty:
    repeated_summary.to_csv(OUTPUT_DIR / "repeated_seed_summary.csv", index=False)

with open(OUTPUT_DIR / "validation.json", "w") as f:
    json.dump(validation, f, indent=2)

with open(OUTPUT_DIR / "experiment_config.json", "w") as f:
    json.dump({
        "dataset_source": selected_repo,
        "temporal_provenance": (
            "row_level_timestamp" if ROW_TIMESTAMP_MODE
            else "capture_day_file_order"
        ),
        "timestamp_policy": (
            "Use a validated actual Timestamp when available; otherwise use "
            "CICIDS2017 capture-day/source-file provenance. Never substitute "
            "Flow Duration, Total TCP Flow Time, packet counts, or IAT features."
        ),
        "quick_validation": QUICK_VALIDATION,
        "paper_mode": PAPER_MODE,
        "fast_train_samples": FAST_TRAIN_SAMPLES,
        "fast_sampling": "stratified" if FAST_SAMPLING_STRATIFIED else "uniform",
        "seeds": list(SEEDS),
        "models": MODEL_NAMES,
        "tasks": ["binary", "multiclass"],
        "missingness_levels": list(MISSINGNESS_LEVELS),
        "conditions": sorted(EXPECTED_CONDITIONS),
        "metrics": [
            "Accuracy", "Macro Precision", "Macro Recall",
            "Macro F1", "Weighted F1", "FPR"
        ],
        "degradation_formula": (
            "(M_reference - M_condition) / M_reference for higher-is-better "
            "metrics; (M_condition - M_reference) / M_reference for FPR"
        ),
        "interaction_formula": (
            "D_combined - (D_temporal + D_missingness)"
        ),
        "confidence_interval": "95% t-based CI across seeds",
        "preprocessing": "median imputation fit on training partition only; StandardScaler fit on training partition only for Logistic Regression",
        "stress_protocol": "random feature-value NaNs injected only into test/evaluation matrices at 10% and 20%, then transformed by the train-fitted imputer",
        "efficiency": [
            "split-level imputer reused across models and tasks",
            "split feature matrices materialized once per seed",
            "fitted estimator reused across normal/10%/20% evaluation",
            "float32 feature matrices",
            "column-wise missingness masks",
            "conservative model parallelism"
        ]
    }, f, indent=2)

with open(OUTPUT_DIR / "dataset_file_validation.json", "w") as f:
    json.dump(file_validation, f, indent=2)

print("Exports written to:", OUTPUT_DIR)


# 27. Run instructions — final validated protocol

### 1. Fast validation first
Leave:

```python
QUICK_VALIDATION = True
PAPER_MODE = False
RUN_EXPERIMENTS = True
```

Then use **Runtime → Run all**.

This validates the complete pipeline with a reproducible stratified bounded training sample:
dataset loading, timestamp provenance, preprocessing, binary/multiclass tasks, all six
reliability conditions, degradation, interaction analysis, attack-level analysis, exports,
and automated validation.

### 2. Final paper experiment
Only after validation passes, change to:

```python
QUICK_VALIDATION = False
PAPER_MODE = True
RUN_EXPERIMENTS = True
```

Then run all cells again.

Paper mode:
- uses the **complete verified training partitions**;
- evaluates binary and multiclass tasks;
- evaluates Reference, Temporal, 10%, 20%, and both combined conditions;
- uses Logistic Regression, Random Forest, and XGBoost;
- runs five seeds: `42, 52, 62, 72, 82`;
- reports mean, standard deviation, and **95% t-based confidence intervals**;
- aggregates degradation and interaction results from the five-seed means;
- aggregates attack-level recall across all five seeds.

### 3. Temporal interpretation
If a valid row-level `Timestamp` exists, the notebook performs a chronological row-level holdout.

If the selected CICIDS2017 mirror does not preserve a trustworthy row-level timestamp, the notebook uses
**capture-day/source-file temporal provenance** and explicitly records that fact.

It never converts `Flow Duration`, `Total TCP Flow Time`, packet counts, IAT values, or any other
flow feature into a timestamp.

### 4. Stress interpretation
The 10% and 20% stress levels represent **random feature-value missingness in the evaluation data**.
The model is trained only once for each model/split/task, and the same fitted estimator is reused across
normal, 10%, and 20% test conditions.

### 5. Final paper outputs
The notebook writes to:

`/content/reliability_outputs/`

including:
- `final_condition_metrics.csv`
- `degradation_macro_f1.csv`
- `degradation_macro_recall.csv`
- `degradation_fpr.csv`
- `interaction_results.csv`
- `attack_level_recall.csv`
- `attack_recall_degradation.csv`
- `repeated_seed_results.csv`
- `repeated_seed_summary.csv`
- six figures
- `validation.json`
- `experiment_config.json`

Do not rewrite the paper's numerical claims until the **PAPER_MODE** run completes with
`Validation status: PASS`.
